# Projekt 2 — Airline Passenger Satisfaction
## Klasyfikacja klasy podróży (Class) — MLP w PyTorch

**Autor:** Dawid Olko  
**Przedmiot:** Sztuczna Inteligencja  
**Prowadzący:** Dr inż. Jacek Bartman  

---

### Cel projektu

Celem jest **klasyfikacja wieloklasowa** klasy podróży pasażerów linii lotniczych na podstawie ocen usług i parametrów lotu.

| Element | Wartość |
|---------|--------|
| Zbiór | Airline Passenger Satisfaction (Kaggle) |
| Target | `Class` (Business, Eco, Eco Plus) |
| Rekordy | ~129 880 (train + test Kaggle) |
| Cechy | 18 numerycznych + 3 kategoryczne |
| Model główny | MLP w PyTorch |
| Model porównawczy | Random Forest |
| Walidacja | 70% / 15% / 15% (stratyfikowana) |
| Metryki | Accuracy, Precision macro, Recall macro, **F1-macro** |

---

### Plan notebooka

1. Eksploracja danych (EDA)
2. Preprocessing — czyszczenie, imputacja, kodowanie, skalowanie
3. Budowa modelu MLP — opis architektury i każdego parametru
4. Trening — optimizer, loss, scheduler, early stopping
5. Grid search hiperparametrów
6. Ewaluacja na zbiorze testowym
7. Porównanie z Random Forest
8. Wnioski

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'airline_project').exists() and (ROOT.parent / 'airline_project').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import json

print('Katalog projektu:', ROOT)
print('Train:', (ROOT / 'data/train.csv').exists())
print('Test:', (ROOT / 'data/test.csv').exists())

## 1. Eksploracja danych (EDA)

Zbiór **Airline Passenger Satisfaction** zawiera dane z ankiet pasażerów.
Każdy wiersz to jeden pasażer z informacjami o:
- profilu (wiek, płeć, typ klienta, cel podróży)
- parametrach lotu (dystans, opóźnienia)
- ocenach usług (skala 0–5: wifi, boarding, rozrywka, komfort, obsługa itd.)

**Target:** `Class` — klasa podróży:
- **Business** (~48%) — klasa biznesowa
- **Eco** (~45%) — klasa ekonomiczna
- **Eco Plus** (~7%) — klasa ekonomiczna plus (mniejszościowa!)

In [ ]:
train_df = pd.read_csv(ROOT / 'data/train.csv')
test_df = pd.read_csv(ROOT / 'data/test.csv')
df = pd.concat([train_df, test_df], ignore_index=True)

print(f'Łączna liczba rekordów: {len(df)}')
print(f'Kolumny: {df.shape[1]}')
print()
print('Rozkład klas (Class):')
print(df['Class'].value_counts())
print()
print('Rozkład procentowy:')
print((df['Class'].value_counts(normalize=True) * 100).round(2))
print()
print('Braki danych (top 5):')
print(df.isna().mean().sort_values(ascending=False).head(5).round(4))

### Kluczowe obserwacje z EDA

1. **Niezbalansowanie klas** — `Eco Plus` to tylko ~7%. Model może "ignorować" tę klasę i wciąż mieć dobrą accuracy globalną.
2. **Braki danych** — jedynie `Arrival Delay in Minutes` ma ~0,3% NaN.
3. **Cechy ocenowe** (skala 0–5) — 14 kolumn z ocenami usług.
4. **Cechy kategoryczne** — 3 kolumny (Gender, Customer Type, Type of Travel).
5. **Kolumna `satisfaction`** — to oryginalny target (satisfied/neutral), ale w naszym projekcie go usuwamy i klasyfikujemy po `Class`.

## 2. Preprocessing

### 2.1 Usuwanie kolumn
- `Unnamed: 0` — sztuczny indeks z pliku CSV
- `id` — identyfikator pasażera (nie niesie informacji)
- `satisfaction` — oryginalny target binarny (nie używamy go)

### 2.2 Usuwanie wartości nierealnych (`remove_unrealistic_values`)

| Kolumna | Warunek na NaN |
|---------|---------------|
| `Age` | < 0 lub > 100 |
| `Flight Distance` | <= 0 |
| Opóźnienia (Departure/Arrival) | < 0 lub > 1440 min (24h) |
| Oceny usług (14 kolumn) | < 0 lub > 5 |

### 2.3 Imputacja braków
- **Numeryczne:** mediana (odporna na wartości odstające)
- **Kategoryczne:** moda (najczęstsza wartość)
- Fit **wyłącznie na zbiorze treningowym** — chroni przed data leakage.

### 2.4 Kodowanie kategorycznych
- **OneHotEncoder** — tworzy kolumny binarne (0/1) dla `Gender`, `Customer Type`, `Type of Travel`
- `handle_unknown='ignore'` — nieznane kategorie na walidacji/teście dają zera

### 2.5 Standaryzacja numerycznych
- **StandardScaler** — transformacja do średniej 0 i odchylenia 1
- Wzór: z = (x − μ) / σ
- Fit **tylko na train** — transform na val i test

### 2.6 Podział danych
- Train: 70%, Validation: 15%, Test: 15%
- **Stratified** — proporcje klas zachowane w każdym podzbiorze
- `random_state=42` — powtarzalność

In [ ]:
from airline_project.preprocessing import przygotuj_surowe_dane, podziel_dane_stratyfikowane, fit_preprocessor, transformuj_cechy
from airline_project.config import TARGET_COLUMN, LABEL_TO_INDEX, CLASS_LABELS

df_clean = przygotuj_surowe_dane(df)
df_clean = df_clean.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

print(f'Po preprocessingu: {len(df_clean)} rekordów')
print(f'Rozkład klas po czyszczeniu:')
print(df_clean[TARGET_COLUMN].value_counts(normalize=True).round(4))

train, val, test = podziel_dane_stratyfikowane(df_clean)
print(f'\nPodział: train={len(train)}, val={len(val)}, test={len(test)}')

artifacts = fit_preprocessor(train)
x_train = transformuj_cechy(train, artifacts)
print(f'Wymiar macierzy cech (train): {x_train.shape}')
print(f'Nazwy cech ({len(artifacts.feature_names)}): {artifacts.feature_names[:10]}...')

## 3. Architektura MLP — szczegółowy opis

### Czym jest MLP (Multi-Layer Perceptron)?

MLP to sieć neuronowa typu **feed-forward** (jednokierunkowa). Dane wchodzą z jednej strony i przechodzą przez kolejne warstwy do wyjścia — nie ma pętli.

### Architektura w tym projekcie

Każda **warstwa ukryta** składa się z 4 bloków (w tej kolejności):

```
Linear → BatchNorm1d → ReLU → Dropout
```

Na końcu jest warstwa **wyjściowa** (sam `Linear`, bez aktywacji — bo CrossEntropyLoss oczekuje surowych logitów).

---

### Opis każdego elementu

#### `nn.Linear(in_features, out_features)`
- **Co robi:** mnożenie macierzowe y = xW^T + b
- **Parametry uczone:** macierz wag W i wektor bias b
- **Sens:** każdy neuron uczy się ważonej kombinacji wejść
- **W projekcie:** np. Linear(24, 512) — z 24 cech tworzy 512 wartości

#### `nn.BatchNorm1d(num_features)`
- **Co robi:** normalizuje wyjścia warstwy w obrębie mini-batcha
- **Wzór:** y = (x - μ_batch) / √(σ²_batch + ε) · γ + β
- **Po co:** stabilizuje i przyspiesza uczenie, zmniejsza wrażliwość na inicjalizację wag
- **γ i β:** parametry uczalne (skala i przesunięcie)

#### `nn.ReLU()`
- **Co robi:** ReLU(x) = max(0, x)
- **Po co:** wprowadza **nieliniowość** — bez niej sieć wielowarstwowa byłaby równoważna jednej warstwie liniowej
- **Zaleta:** prostota i szybkość; nie ma problemu zanikającego gradientu (w dodatnich wartościach)

#### `nn.Dropout(p)`
- **Co robi:** losowo zeruje p% neuronów podczas treningu
- **Po co:** **regularyzacja** — zapobiega przeuczeniu (overfitting)
- **W projekcie:** p = 0.1, 0.2 lub 0.3 (testowane w grid search)
- **Na walidacji/teście:** dropout jest wyłączony (model.eval())

---

### Warianty architektury (grid search)

| Architektura | Warstwy ukryte | Liczba parametrów (rząd wielkości) |
|-------------|----------------|------------------------------------|
| (128, 64) | 2 warstwy | ~12 tys. |
| (256, 128, 64) | 3 warstwy | ~50 tys. |
| (512, 256, 128) | 3 warstwy | ~200 tys. |

Większa sieć = więcej pojemności, ale ryzyko overfittingu (dlatego Dropout!).

In [ ]:
from airline_project.model import AirlineMLP, wybierz_urzadzenie
import torch

device = wybierz_urzadzenie()
print(f'Urządzenie: {device}')

example_model = AirlineMLP(
    input_dim=24,
    hidden_sizes=(512, 256, 128),
    output_dim=3,
    dropout=0.3,
)
print('\nArchitektura modelu:')
print(example_model)
print(f'\nŁączna liczba parametrów: {sum(p.numel() for p in example_model.parameters()):,}')

## 4. Proces treningu — opis parametrów

### 4.1 Optimizer: AdamW

- **Adam** = Adaptive Moment Estimation — łączy momentum (średnia ruchoma gradientów) z adaptacyjnym krokiem (RMSProp)
- **AdamW** = Adam z **poprawioną regularyzacją L2** (weight decay jest niezależny od learning rate)
- `learning_rate=1e-3` — wielkość kroku w kierunku minimum
- `weight_decay=1e-4` — kara za duże wagi (ogranicza złożoność modelu)

### 4.2 Loss function: CrossEntropyLoss z class weights

- **CrossEntropyLoss** = softmax + negative log-likelihood
- Dla próbki klasy k: loss = -log(P(klasa=k))
- **class_weights** — mnożnik straty per klasa: rzadka klasa (Eco Plus) ma wyższy koszt błędu
- Wagi liczone przez `compute_class_weight('balanced')` — odwrotnie proporcjonalne do częstości

### 4.3 Scheduler: ReduceLROnPlateau

- **Co robi:** gdy metryka (val macro F1) przestaje rosnąć, zmniejsza learning rate o połowę
- `mode='max'` — monitorujemy wzrost F1 (nie spadek loss)
- `factor=0.5` — nowy LR = stary LR × 0.5
- `patience=3` — czeka 3 epoki bez poprawy zanim zmniejszy

### 4.4 Early stopping

- Monitorujemy **walidacyjne macro F1**
- Jeśli nie poprawi się o `min_delta=0.0001` przez `patience=5` epok → STOP
- Zapisujemy najlepszy checkpoint (model z najwyższym val F1)
- Chroni przed overfittingiem i skraca czas treningu

### 4.5 Batch size i epoki

- **Batch size = 1024** — duży batch bo duży zbiór (~90k train). Szybsze epoki, stabilniejsze gradienty.
- **Max epochs = 30–35** — z early stopping zwykle zatrzymuje się wcześniej.

## 5. Grid search hiperparametrów

Systematycznie przeszukujemy:

| Parametr | Wartości |
|----------|----------|
| Architektura | (128, 64), (256, 128, 64), (512, 256, 128) |
| Dropout | 0.1, 0.2, 0.3 |
| Learning rate | 1e-3, 5e-4 |

**3 × 3 × 2 = 18 konfiguracji**

Każda trenowana z early stopping. Najlepsza po walidacyjnym macro F1 = **tuned**.
Stała konfiguracja (128, 64), dropout=0.2, lr=1e-3 = **baseline**.

In [ ]:
grid_results = pd.read_csv(ROOT / 'results/airline/grid_search_results.csv')
print('Top 5 konfiguracji z grid search:')
print(grid_results[['name', 'best_val_f1_macro', 'dropout', 'learning_rate']].head(5).to_string(index=False))
print(f'\nNajlepsza: {grid_results.iloc[0]["name"]}')
print(f'Walidacyjne macro F1: {grid_results.iloc[0]["best_val_f1_macro"]:.4f}')

## 6. Ewaluacja na zbiorze testowym

Po grid search trenujemy ponownie:
- **baseline** (MLP 128×64, dropout=0.2, lr=1e-3)
- **tuned** (najlepsza konfiguracja z siatki)
- **Random Forest** (domyślne parametry sklearn, 100 drzew)

Oceniamy na **teście (15%)** — dane nigdy niewidziane w treningu ani walidacji.

In [ ]:
comparison = pd.read_csv(ROOT / 'results/airline/model_comparison.csv')
print('Tabela porównawcza modeli na zbiorze testowym:')
print(comparison.round(4).to_string(index=False))

In [ ]:
for name in ['mlp_baseline', 'mlp_tuned', 'random_forest']:
    path = ROOT / f'results/airline/classification_report_{name}.txt'
    if path.exists():
        print(f'\n{"=" * 50}')
        print(f'Classification report: {name}')
        print('=' * 50)
        print(path.read_text())

## 7. Wykresy

Poniżej wyświetlamy wykresy wygenerowane podczas eksperymentu.

In [ ]:
from IPython.display import Image, display

plots_dir = ROOT / 'results/airline/plots'
for plot_file in ['loss_and_f1_history.png', 'scheduler_lr.png', 'confusion_matrices_side_by_side.png']:
    path = plots_dir / plot_file
    if path.exists():
        print(f'\n--- {plot_file} ---')
        display(Image(filename=str(path), width=800))

## 8. FAQ — wyjaśnienie prostym językiem

### Jak działa train / val / test? (najważniejsze)

**Tak — ogólnie dobrze:** uczysz na 70%, kontrolujesz na 15% val, a **dopiero na końcu** oceniasz na 15% test.

1. **Train (70%)** — model uczy się (zmienia wagi).
2. **Val (15%)** — **nie** jest testem końcowym. Co epokę sprawdzamy wynik: early stopping, scheduler LR, wybór najlepszego wariantu z grid search.
3. **Test (15%)** — **jeden raz na końcu**, gdy model już wybrany. Stąd raporty z accuracy 0.76 / 0.77 i 19483 próbek.

Val = wiele kontroli w trakcie nauki. Test = egzamin końcowy.

### Co to jest wyciek danych (data leakage)?

**Wyciek** = test lub odpowiedź „przedostaje się” do uczenia → wynik na teście jest **zawyżony** (model miał podpowiedź).

**Przykłady wycieku:**
- liczenie średniej/mediana na train+test razem (zamiast fit tylko na train),
- zostawienie `satisfaction` w cechach (podpowiedź związana z `Class`),
- wybór najlepszego modelu po **test** zamiast po **val**.

**W projekcie chronimy się tak:**
- `fit_preprocessor` tylko na train,
- `transform` na val/test,
- grid search / early stopping na val,
- test tylko na końcu (raz).

Kod: `preprocessing.py` → `fit_preprocessor(train_df)`; `experiment.py` → ewaluacja końcowa na teście.

### Dlaczego usuwamy `Unnamed: 0`, `id`, `satisfaction`?
- `Unnamed: 0` to techniczny numer wiersza z CSV, nie cecha pasażera.
- `id` to identyfikator osoby, model nie powinien uczyć się numerów.
- `satisfaction` to inny target (zadowolony/niezadowolony). W tym projekcie targetem jest `Class`, więc `satisfaction` usuwamy, żeby nie mieszać zadań.

### Co znaczy: "fit na train, transform na val/test"?
- **fit** = "naucz parametry preprocessingu" (np. średnia, odchylenie, mediana, słownik kategorii).
- **transform** = "użyj już nauczonych parametrów".
- Dlaczego tak? Bo gdybyśmy liczyli parametry na val/test, to model dostałby podpowiedź z danych, których nie powinien znać.

### Po co `handle_unknown='ignore'`?
Jeśli w walidacji/test pojawi się nowa kategoria (np. literówka lub nowa wartość), kod się nie wywali. Taka nieznana kategoria dostanie zera w kolumnach one-hot.

### Dlaczego podział 70/15/15, a nie 80/20?
- 80/20 jest OK, gdy nie stroisz modelu.
- Tu stroimy (grid search + early stopping), więc potrzebujemy osobnego zbioru **walidacyjnego**.
- Dlatego:
  - 70% train — uczenie wag,
  - 15% val — wybór hiperparametrów,
  - 15% test — uczciwa końcowa ocena.

### Czemu widzę 25k w `test.csv`, a w wynikach ~19.5k testu?
Bo łączymy `data/train.csv` i `data/test.csv` Kaggle w jedną ramkę, a potem robimy nowy podział 70/15/15. Stąd finalny test to ~15% całości, czyli ok. 19 483.

### Co to jest warstwa: `Linear -> BatchNorm -> ReLU -> Dropout`?
- `Linear` — liczy ważoną sumę cech (to podstawowy "blok obliczeń").
- `BatchNorm` — stabilizuje wartości między warstwami.
- `ReLU` — dodaje nieliniowość (bez niej sieć byłaby zbyt prosta).
- `Dropout` — losowo wyłącza część neuronów w treningu, żeby model nie "zakuwał" danych na pamięć.

### Co to jest `output_dim=3`?
Mamy 3 klasy targetu `Class`: `Business`, `Eco`, `Eco Plus`. Ostatnia warstwa musi mieć tyle neuronów, ile klas — więc 3.

### Co zmienia liczba warstw ukrytych i ich rozmiar?
- Więcej / większe warstwy = model może nauczyć się trudniejszych zależności.
- Ale zbyt duży model łatwiej się przeucza.
- Dlatego testujemy kilka wariantów i wybieramy najlepszy na walidacji.

### Co to jest grid search?
Automatyczne sprawdzenie wielu kombinacji parametrów (architektura, dropout, learning rate). Wybieramy konfigurację z najlepszym wynikiem na walidacji.

### Co to jest `CrossEntropyLoss`?
To funkcja straty dla klasyfikacji wieloklasowej. Karze model, gdy daje niskie prawdopodobieństwo prawdziwej klasy.

### Co to są class weights?
Ważą klasy odwrotnie do ich liczności. Rzadka klasa (`Eco Plus`) dostaje większą "karę za błąd", więc model bardziej się na niej skupia.

### Co to jest F1-macro?
F1 liczony osobno dla każdej klasy, a potem zwykła średnia. Każda klasa ma równą wagę. Dlatego F1-macro jest dobre przy niezbalansowanych danych.

### Co to jest SMOTE i oversampling?
- **Oversampling**: sztuczne zwiększanie liczby próbek klasy mniejszościowej.
- **SMOTE**: tworzy nowe, syntetyczne próbki tej klasy.
W tym projekcie celowo tego nie używamy (wymaganie), zamiast tego używamy class weights.

### Czym różnią się metryki?
- `accuracy` — procent wszystkich trafień.
- `precision_macro` — jak często model ma rację, gdy wskazuje daną klasę.
- `recall_macro` — ile prawdziwych przypadków danej klasy model wykrył.
- `f1_macro` — kompromis precision/recall, średnio po klasach.

### Co znaczy "tryb eval"?
`model.eval()` to tryb oceny: dropout jest wyłączony, batchnorm pracuje stabilnie. Używamy go na walidacji i teście.

### Co to są: AdamW, ReduceLROnPlateau, early stopping?
- **AdamW** — algorytm aktualizacji wag.
- **ReduceLROnPlateau** — gdy wynik przestaje się poprawiać, zmniejsza learning rate.
- **Early stopping** — zatrzymuje trening, gdy brak poprawy przez kilka epok.

### Co to jest epoka?
Jedna epoka = model zobaczył wszystkie próbki treningowe raz.

## 9. Wnioski końcowe

### 9.1 Najlepszy wynik

Na zbiorze testowym (19 483 próbki) najlepszy **F1-macro** osiągnął **MLP tuned** (~0.653).
Random Forest ma wyższą **accuracy** (~0.864), ale **niższe F1-macro** (~0.606) — słabo rozpoznaje klasę Eco Plus.

### 8.2 Porównanie MLP vs Random Forest

| Model | Accuracy | F1-macro | Uwagi |
|-------|----------|----------|-------|
| MLP baseline | 0.773 | 0.650 | Prostsza sieć (128→64) |
| **MLP tuned** | 0.783 | **0.653** | Głębsza sieć (512→256→128), dropout=0.3 |
| Random Forest | **0.864** | 0.606 | Wysoka accuracy, ale ignoruje Eco Plus |

**Kluczowy wniosek:** RF wygrywa accuracy, ale **MLP wygrywa F1-macro** dzięki class weights — RF domyślnie nie balansuje klas i prawie nie rozpoznaje Eco Plus (recall ~1%).

### 8.3 Interpretacja macierzy pomyłek

- **Business** — dobrze rozpoznawany przez wszystkie modele (precision ~95%)
- **Eco** — dobrze rozpoznawany, ale mylony z Business
- **Eco Plus** (problem!):
  - MLP tuned: recall ~41% (łapie mniej niż połowę, ale przynajmniej próbuje)
  - Random Forest: recall ~1% (prawie całkowicie ignoruje tę klasę!)
  - To dlatego, że Eco Plus to tylko 7% danych

### 8.4 Dlaczego F1-macro jest ważniejsze niż accuracy?

- Accuracy = % wszystkich trafień (zdominowane przez duże klasy)
- F1-macro = średnia F1 **z równymi wagami** per klasa
- Model z accuracy 86% ale ignorujący 7% populacji nie jest "lepszy" w praktyce

### 8.5 Wpływ class weights

MLP używa `CrossEntropyLoss(weight=...)` — wyższy koszt błędu na Eco Plus zmusza sieć do zwracania uwagi na tę klasę. RF bez class weights po prostu klasyfikuje wszystko jako Business/Eco.

### 8.6 Ograniczenia projektu

1. **Klasa Eco Plus jest bardzo mała** (~7%) — żaden model nie osiąga świetnych wyników na niej
2. **Dane tabelaryczne** — MLP nie wykorzystuje czasowości ani interakcji wyższego rzędu tak dobrze jak modele drzewiaste
3. **Grid search ograniczony do 18 konfiguracji** — kompromis czasowy; pełny random search mógłby znaleźć lepsze punkty
4. **Brak feature engineering** — np. suma ocen usług, ratio opóźnień do dystansu
5. **Target `Class` vs `satisfaction`** — Class ma inny sens niż oryginalna zmienna satisfaction

### 8.7 Możliwe ulepszenia

- Dodanie RF z `class_weight='balanced'` do porównania
- Oversampling Eco Plus (np. SMOTE) zamiast class weights
- Szerszy grid search lub bayesowska optymalizacja hiperparametrów
- Inżynieria cech: kombinacje, agregaty, PCA
- Ensemble: MLP + RF jako voting/stacking

## 10. Dokładne raporty klasyfikacji i ich interpretacja

Poniżej dokładne wyniki per klasa (te same, które masz w plikach `classification_report_*.txt`).

### MLP baseline

```text
              precision    recall  f1-score   support

    Business       0.94      0.85      0.89      9325
         Eco       0.81      0.73      0.77      8747
    Eco Plus       0.18      0.42      0.26      1411

    accuracy                           0.76     19483
   macro avg       0.65      0.67      0.64     19483
weighted avg       0.83      0.76      0.79     19483
```

### MLP tuned

```text
              precision    recall  f1-score   support

    Business       0.94      0.85      0.89      9325
         Eco       0.81      0.75      0.78      8747
    Eco Plus       0.19      0.41      0.26      1411

    accuracy                           0.77     19483
   macro avg       0.65      0.67      0.65     19483
weighted avg       0.83      0.77      0.80     19483
```

### Random Forest

```text
              precision    recall  f1-score   support

    Business       0.94      0.91      0.93      9325
         Eco       0.80      0.95      0.87      8747
    Eco Plus       0.30      0.01      0.02      1411

    accuracy                           0.86     19483
   macro avg       0.68      0.62      0.61     19483
weighted avg       0.83      0.86      0.84     19483
```

### Jak to interpretować?

- **Business:** wszystkie modele bardzo dobre.
- **Eco:** Random Forest ma najwyższy recall (0.95), ale „przesadza” w stronę Eco.
- **Eco Plus:** tu widać największy problem:
  - MLP: recall ~0.41-0.42 (wykrywa około 4/10 przypadków)
  - RF: recall 0.01 (praktycznie nie wykrywa)
- Dlatego:
  - RF ma najlepszą accuracy (bo trafia duże klasy),
  - MLP tuned ma lepszy F1-macro (bardziej sprawiedliwy dla wszystkich klas).

### Co oznacza wykres LR (`scheduler_lr.png`)?

- LR = learning rate = wielkość kroku uczenia.
- Na początku LR jest większy, żeby szybciej się uczyć.
- Kiedy walidacyjne F1 przestaje rosnąć, scheduler zmniejsza LR (najczęściej o połowę).
- Na wykresie widać schodki w dół.
- To jest dobre: model najpierw uczy się szybko, a potem dokładnie „dostraja”.

In [ ]:
meta = json.loads((ROOT / 'results/airline/run_metadata.json').read_text())
print('Metadane uruchomienia:')
print(f"  Urządzenie: {meta['device']}")
print(f"  Rekordy: {meta['n_records']}")
print(f"  Train/Val/Test: {meta['n_train']}/{meta['n_val']}/{meta['n_test']}")
print(f"  Cech po preprocessingu: {meta['n_features']}")
print(f"  Tuned config: {meta['tuned_config']['name']}")
print(f"  Best val F1: {meta['best_grid']['best_val_f1_macro']:.4f}")